In [ ]:
from Code.NetworkExpansion import *
from Code.externalize_results import *
import networkx as nx
import matplotlib.pyplot as plt
from networkx.algorithms import bipartite

# Expansions from primordial seed sets
write_outputs = True
max_generation = 80
name = 'primordial'

# define chemical scope
reaction_scope = get_sides(import_reaction_file()) # camerian todo: update w/latest commit
compound_scope = set.union(*reaction_scope.stack())
seed_scope = seed_candidates.loc[seed_candidates.index.isin(compound_scope)]
seeds = list(seed_scope.index)

# run network expansion
result = expand_from_seed_set(seeds=seeds, reactions=reaction_scope, max_gen=max_generation)
write_expansion_csv(result, filename=name, destination_dir='Output/'+name+'/')
result = generate_annotated_bipartite_edgelist(result)
summary = generate_expansion_summary(result)
transition_gen = summary['chiral fraction'].where(lambda x: x>0.5).idxmin()
transition_edge_df = result['bipartite_edgelist'].query('step <= @transition_gen').copy(deep=True)
networks = {i: nx.from_pandas_edgelist(transition_edge_df[transition_edge_df[i]], 
        source='source', target='target', edge_attr=True, create_using=nx.DiGraph())
        for i in ['impactful_precursor','immediate_precursor','directed']}
compound_nodes = {i: set(j.nodes()).difference(reaction_scope.index) for i,j in networks.items()}
networks_uni = {i: bipartite.projected_graph(networks[i], nodes=compound_nodes[i]) for i in networks.keys()}
